# Signal

> Signal Processing Helper Functions

In [ ]:
#| default_exp signal

In [ ]:
#| hide
from nbdev.showdoc import *
import glob, zarr

In [ ]:
#| export
from scipy import signal, interpolate
import torch, numpy as np, torch.nn.functional as F
from fractions import Fraction

In [ ]:
#| export
def butterworth(waveform_array, # waveform array of shape (n_samples,)
                freq_range, # e.g. [0.5, 8] for bandpass, 0.5 for highpass, 8 for lowpass
                btype, # 'bandpass', 'lowpass', 'highpass', 'bandstop'
                fs=128, # sampling frequency
                order=4 # filter order (default 4)
                ): 
    """
    Butterworth filter

    Returns: filtered waveform array of shape (n_samples,)
    """
    sos = signal.butter(order, freq_range, fs=fs, btype=btype, output='sos')
    filtered = signal.sosfiltfilt(sos, waveform_array) # zero phase filter (no phase shift)
    return filtered

def resample_waveform(waveform_array, # waveform array of shape (n_samples,)
                      fs_in, # original sampling frequency
                      fs_out, # desired sampling frequency
                      is_spo2=False # if True, use linear interpolation (for SpO2 signal or other low-sampling-rate signals)
                      ):
    """
    Resample waveform to desired sampling frequency
    
    Returns: resampled waveform array of shape (n_resampled_samples,)
    """
    if not is_spo2:
        resample_fraction = Fraction(fs_out, fs_in)#.limit_denominator(100)
        resampled_waveform = signal.resample_poly(waveform_array, resample_fraction.numerator, resample_fraction.denominator)
    else:
        # linear interpolation
        t = np.arange(0, len(waveform_array)*(1/fs_in), 1/fs_in)
        resample_f = interpolate.make_interp_spline(t, waveform_array, k=1) # linear interpolation
        t_new = np.arange(0, len(waveform_array)*(1/fs_in), 1/fs_out)
        resampled_waveform = resample_f(t_new)
    return resampled_waveform


def iir_filter(waveform_array, # waveform array of shape (n_samples,),
               freq_range, # e.g. [0.5, 8] for bandpass, 0.5 for highpass, 8 for lowpass
               btype,  # 'bandpass', 'lowpass', 'highpass', 'bandstop'
               order=16,  # filter order (default 16)
               fs=128  # sampling frequency (default 128)
               ):
    """
    IIR filter using elliptic filter design
    as described in https://www.researchsquare.com/article/rs-6307069/v1

    Returns: filtered waveform array of shape (n_samples,)
    """
    sos = signal.iirfilter(N=order, Wn=freq_range, rp=1, rs=40, btype=btype, analog=False, ftype='ellip', output='sos', fs=fs)
    filtered_data = signal.sosfiltfilt(sos, waveform_array)
    return filtered_data

def iqr_normalization(waveform_array, # waveform array of shape (n_samples,)
                      is_spo2=False # if True, use SpO2 normalization (0.6-1.0 scaled to -1 to 1)
                      ):
    """
    IQR normalization to scale waveform to -1 to 1
    For SpO2, scale 0.6-1.0 to -1 to 1
    
    Returns: normalized waveform array of shape (n_samples,)
    """
    eps = 1e-10
    if not is_spo2:
        q5 = np.percentile(waveform_array, 5)
        q95 = np.percentile(waveform_array, 95)
        waveform_array = 2*(waveform_array - q5) / (q95 - q5 + eps) - 1 # scaled to -1 to 1
    else:
        # spo2 is scaled to 0.60-1.0
        if np.mean(waveform_array) > 1.0:
            waveform_array = waveform_array / 100.
        waveform_array = waveform_array.clip(0.6, 1.0)
        waveform_array = 2*(waveform_array - 0.6) / (1.0 - 0.6) - 1 # scaled to -1 to 1
    return waveform_array

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()